# Scraping Google search results: AI Copyright & Governance

---
## 0. Install Dependencies

In [ ]:
!pip install google-search-results pandas python-dotenv requests trafilatura youtube_transcript_api

---
## 1. Imports & Configuration

In [16]:
from serpapi import GoogleSearch
from youtube_transcript_api import YouTubeTranscriptApi
from concurrent.futures import ThreadPoolExecutor, as_completed
import trafilatura
import pandas as pd
from dotenv import load_dotenv
import os
import re
import time

load_dotenv()
SERP_API_KEY = os.getenv("SERPAPI_KEY")

# Max number of articles to scrape
MAX_NUM_RESULTS = 10

---
## 2. SerpApi - Collecting article & video URLs

In [17]:
def collect_search_results(queries: list[str],
                           num_results: int = MAX_NUM_RESULTS) -> tuple[pd.DataFrame, list[dict]]:
    """
    Search Google for each query and collect:
    - organic_results  → articles (title, snippet, url, source, date)
    - inline_videos    → YouTube videos (title, channel, url)

    Returns:
        articles_df  : DataFrame of all article metadata
        raw_results  : list of raw SerpAPI result dicts (just for inspection)
    """
    article_rows = []
    video_rows   = []
    raw_results  = []

    for query in queries:
        print(f"\n🔍 Searching: '{query}'")

        params = {
            "engine":        "google",
            "q":             query,
            "google_domain": "google.com",
            "hl":            "en",
            "gl":            "us",
            "api_key":       SERP_API_KEY
        }

        search  = GoogleSearch(params)
        results = search.get_dict()
        raw_results.append({"query": query, "results": results})

        # ── Organic articles ──
        organic = results.get("organic_results", [])
        if not organic:
            print(f"  ⚠️  No organic_results for: {query}")
        else:
            for article in organic[:num_results]:
                article_rows.append({
                    "query":   query,
                    "title":   article.get("title"),
                    "snippet": article.get("snippet"),
                    "source":  article.get("source"),
                    "date":    article.get("date"),
                    "url":     article.get("link"),
                    "type":    "article"
                })
            print(f"  ✅ {len(organic[:num_results])} articles collected")

        # ── Inline videos ──
        videos = results.get("inline_videos", [])
        if videos:
            for video in videos:
                video_rows.append({
                    "query":   query,
                    "title":   video.get("title"),
                    "snippet": None,
                    "source":  video.get("channel"),
                    "date":    None,
                    "url":     video.get("link"),
                    "type":    "video"
                })
            print(f"  ✅ {len(videos)} videos collected")

    # Combine into single DataFrame
    all_rows = article_rows + video_rows
    df = pd.DataFrame(all_rows)

    if df.empty:
        print("\n⚠️  No results collected across all queries.")
        return df, raw_results

    # Remove duplicates by URL
    df = df.drop_duplicates(subset="url").reset_index(drop=True)

    return df, raw_results

In [18]:
# ── Collect urls from search query ──
df, raw = collect_search_results(["AI intellectual property", 
                                  "copyright Generative AI"])
df.head()


🔍 Searching: 'AI intellectual property'
  ✅ 10 articles collected

🔍 Searching: 'copyright Generative AI'
  ✅ 10 articles collected


,query,title,snippet,source,date,url,type
0,AI intellectual property,Generative AI: Navigating intellectual property,Fully AI-generated content is ineligible for c...,Nixon Peabody,"Sep 17, 2025",https://www.nixonpeabody.com/insights/articles...,article
1,AI intellectual property,Artificial Intelligence and Intellectual Property,AI inventions present the current patent syste...,World Intellectual Property Organization (WIPO),NaN,https://www.wipo.int/en/web/frontier-technolog...,article
2,AI intellectual property,"AI, Copyright, and the Law: The Ongoing Battle...","In the past year, courts, legislators, and reg...",University of Southern California,"Feb 4, 2025",https://sites.usc.edu/iptls/2025/02/04/ai-copy...,article
3,AI intellectual property,Copyright Or Copywrong? AI's Intellectual Prop...,The report states that AI-generated works need...,Forbes,"Feb 11, 2025",https://www.forbes.com/sites/douglaslaney/2025...,article
4,AI intellectual property,AI and intellectual property rights,AI tools are now increasingly being used for s...,Dentons,"Jan 28, 2025",https://www.dentons.com/en/insights/articles/2...,article


In [20]:
raw

[{'query': 'AI intellectual property',
  'results': {'search_metadata': {'id': '69ce4faf2771430db9186b92',
    'status': 'Success',
    'json_endpoint': 'https://serpapi.com/searches/G3XbEcf9TUuxjG1sMWYgSg/69ce4faf2771430db9186b92.json',
    'pixel_position_endpoint': 'https://serpapi.com/searches/G3XbEcf9TUuxjG1sMWYgSg/69ce4faf2771430db9186b92.json_with_pixel_position',
    'created_at': '2026-04-02 11:14:55 UTC',
    'processed_at': '2026-04-02 11:14:55 UTC',
    'google_url': 'https://www.google.com/search?q=AI+intellectual+property&oq=AI+intellectual+property&hl=en&gl=us&sourceid=chrome&ie=UTF-8',
    'raw_html_file': 'https://serpapi.com/searches/G3XbEcf9TUuxjG1sMWYgSg/69ce4faf2771430db9186b92.html',
    'total_time_taken': 0.7},
   'search_parameters': {'engine': 'google',
    'q': 'AI intellectual property',
    'google_domain': 'google.com',
    'hl': 'en',
    'gl': 'us',
    'device': 'desktop'},
   'search_information': {'query_displayed': 'AI intellectual property',
    'to

---
## 3. Scraping texts from articles and videos

In [21]:
# Define scrapers
def _scrape_url(url: str) -> dict:
    """Scrape clean article text from a URL using Trafilatura."""
    try:
        downloaded = trafilatura.fetch_url(url)
        if not downloaded:
            return {"url": url, "full_text": None,
                    "status": "failed_download"}

        text = trafilatura.extract(
            downloaded,
            include_comments=False,
            include_tables=True,
            no_fallback=False,
            favor_precision=False,
            deduplicate=True
        )
        return {
            "url":       url,
            "full_text": text.strip() if text else None,
            "status":    "success" if text else "failed_extraction"
        }
    except Exception as e:
        return {"url": url, "full_text": None,
                "status": f"error: {str(e)}"}


def _get_youtube_id(url: str) -> str | None:
    """Extract YouTube video ID from URL."""
    patterns = [
        r"youtube\.com/watch\?v=([a-zA-Z0-9_-]{11})",
        r"youtube\.com/shorts/([a-zA-Z0-9_-]{11})",
        r"youtu\.be/([a-zA-Z0-9_-]{11})"
    ]
    for pattern in patterns:
        match = re.search(pattern, url)
        if match:
            return match.group(1)
    return None


def _get_transcript(video_id: str) -> str | None:
    """Fetch YouTube transcript from video ID."""
    try:
        transcript = YouTubeTranscriptApi.get_transcript(
            video_id,
            languages=["en", "en-US", "en-GB"]
        )
        return " ".join([t["text"] for t in transcript])
    except Exception:
        return None


# ═══════════════════════════════════════════════════════
# ENRICHMENT
# ═══════════════════════════════════════════════════════

def enrich_search_results(df: pd.DataFrame,
                         max_workers: int = 5,
                         delay: float = 1.0) -> pd.DataFrame:
    """
    Enrich a DataFrame from collect_search_results() with full text content.
    - Rows with type='article' → scraped via Trafilatura
    - Rows with type='video'   → transcript via YouTube Transcript API

    Args:
        df:          DataFrame returned by collect_articles()
        max_workers: concurrent threads for article scraping
        delay:       seconds between requests per thread

    Returns:
        Enriched DataFrame with added columns:
        - full_text  : scraped article text OR video transcript
        - video_id   : YouTube video ID (videos only)
        - status     : success / failed_download / no_transcript / etc.
    """
    if df.empty:
        print("⚠️  Empty DataFrame — nothing to enrich.")
        return df

    df = df.copy()
    df["full_text"] = None
    df["video_id"]  = None
    df["status"]    = "pending"

    # ── Split by type ──
    article_mask = df["type"] == "article"
    video_mask   = df["type"] == "video"

    article_df = df[article_mask].copy()
    video_df   = df[video_mask].copy()

    # ── Scrape articles ──
    if not article_df.empty:
        print(f"\n📰 Scraping {len(article_df)} articles...")

        url_to_result = {}

        def fetch_with_delay(url):
            time.sleep(delay)
            return _scrape_url(url)

        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = {
                executor.submit(fetch_with_delay, row["url"]): idx
                for idx, row in article_df.iterrows()
            }
            for i, future in enumerate(as_completed(futures)):
                result = future.result()
                url_to_result[result["url"]] = result
                icon = "✅" if result["status"] == "success" else "❌"
                source = df.loc[futures[future], "source"] or "Unknown"
                print(f"  {icon} [{i+1}/{len(article_df)}] "
                      f"{source} — {result['status']}")

        # Map results back
        for idx, row in article_df.iterrows():
            r = url_to_result.get(row["url"], {})
            df.at[idx, "full_text"] = r.get("full_text")
            df.at[idx, "status"]    = r.get("status", "unknown")

        success = sum(
            1 for r in url_to_result.values()
            if r["status"] == "success"
        )
        print(f"\n  📊 Articles: {success}/{len(article_df)} "
              f"scraped successfully")

    # ── Fetch video transcripts ──
    if not video_df.empty:
        print(f"\n🎬 Fetching transcripts for "
              f"{len(video_df)} videos...")

        for idx, row in video_df.iterrows():
            url      = row["url"]
            vid_id   = _get_youtube_id(url)
            transcript = None

            if vid_id:
                transcript = _get_transcript(vid_id)

            status = "success" if transcript else "no_transcript"
            icon   = "✅" if transcript else "❌"

            df.at[idx, "video_id"]  = vid_id
            df.at[idx, "full_text"] = transcript
            df.at[idx, "status"]    = status

            title = (row["title"] or "")[:50]
            print(f"  {icon} {title}... [{status}]")

        success = sum(
            1 for idx in video_df.index
            if df.at[idx, "status"] == "success"
        )
        print(f"\n  📊 Videos: {success}/{len(video_df)} "
              f"transcripts fetched")

    return df

In [22]:
# ── Enrich data with full texts from articles and videos ──
df = enrich_search_results(df)

# ── Filter to only successfully scraped content ──
df_clean = df[df["status"] == "success"].reset_index(drop=True)

# ── Inspect results ──
df_clean.head()


📰 Scraping 18 articles...
  ❌ [1/18] Forbes — failed_download
  ❌ [2/18] Dentons — failed_download
  ✅ [3/18] World Intellectual Property Organization (WIPO) — success
  ✅ [4/18] Nixon Peabody — success
  ✅ [5/18] University of Southern California — success
  ❌ [6/18] American Bar Association — failed_download
  ✅ [7/18] DarrowEverett LLP — success
  ✅ [8/18] Copyright Office (.gov) — success
  ✅ [9/18] Gowling WLG — success
  ❌ [10/18] Congress.gov — failed_download
  ✅ [11/18] Ironhack — success
  ✅ [12/18] The Regulatory Review — success
  ✅ [13/18] University of South Florida — success
  ✅ [14/18] Copyright Office (.gov) — success
  ❌ [15/18] RAND — failed_download
  ✅ [16/18] Wikipedia — success
  ✅ [17/18] Yale Law Journal — success
  ✅ [18/18] Association of Research Libraries — success

  📊 Articles: 13/18 scraped successfully


,query,title,snippet,source,date,url,type,full_text,video_id,status
0,AI intellectual property,Generative AI: Navigating intellectual property,Fully AI-generated content is ineligible for c...,Nixon Peabody,"Sep 17, 2025",https://www.nixonpeabody.com/insights/articles...,article,Generative AI is transforming creative and tec...,None,success
1,AI intellectual property,Artificial Intelligence and Intellectual Property,AI inventions present the current patent syste...,World Intellectual Property Organization (WIPO),NaN,https://www.wipo.int/en/web/frontier-technolog...,article,Artificial Intelligence and Intellectual Prope...,None,success
2,AI intellectual property,"AI, Copyright, and the Law: The Ongoing Battle...","In the past year, courts, legislators, and reg...",University of Southern California,"Feb 4, 2025",https://sites.usc.edu/iptls/2025/02/04/ai-copy...,article,By: Negar Bondari\nArtificial intelligence (AI...,None,success
3,AI intellectual property,AI Created It—But Do You Own It? IP Issues Exp...,AI-generated content raises complex legal ques...,DarrowEverett LLP,"Jul 7, 2025",https://darroweverett.com/ai-and-the-law-who-o...,article,Legal Insights\nAs artificial intelligence (AI...,None,success
4,AI intellectual property,Copyright and Artificial Intelligence | U.S. C...,Copyright and Artificial Intelligence analyzes...,Copyright Office (.gov),NaN,https://www.copyright.gov/ai/,article,Copyright and Artificial Intelligence\nSince l...,None,success


In [ ]:
df_clean

In [23]:
df_clean.to_csv("ai_copyright_dataset.csv", index=False)